In [25]:
import pandas as pd
import numpy as np
import tkinter as tk

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from tkinter.filedialog import askopenfilename
from sklearn.metrics import confusion_matrix, classification_report


# from matplotlib import pyplot
# import seaborn as sns

# from sklearn.compose import ColumnTransformer
# from sklearn.impute import SimpleImputer
# from sklearn.preprocessing import OneHotEncoder, MinMaxScaler, StandardScaler, OrdinalEncoder
# from sklearn.pipeline import Pipeline,make_pipeline
# from sklearn.feature_selection import SelectKBest,chi2
# from sklearn.tree import DecisionTreeClassifier
# from sklearn.ensemble import RandomForestClassifier
# from sklearn.model_selection import train_test_split
# from sklearn.linear_model import LogisticRegression
# from sklearn.ensemble import GradientBoostingRegressor
# from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score,classification_report,confusion_matrix
# from imblearn.pipeline import Pipeline as ImbPipeline
# from imblearn.over_sampling import SMOTE
# from matplotlib import pyplot as plt
# from xgboost import XGBClassifier



In [26]:

root = tk.Tk()
root.withdraw()
root.attributes("-topmost", True)

file_path = askopenfilename(parent=root, title="Select CSV or Excel file")

root.destroy()

if file_path:
    if file_path.lower().endswith(".csv"):
        df = pd.read_csv(file_path)
    else:
        df = pd.read_excel(file_path, engine="openpyxl")
    #print(df.head())
else:
    print("No file selected.")
    
#df = pd.read_csv ('data/Customer_Loan_Request.csv')
#data = pd.read_csv ('data/Customer_Loan_Request.csv')

df.head (3) 

,CustomerID,Name,Age,Gender,MaritalStatus,EducationLevel,EmploymentStatus,AnnualIncome,LoanAmountRequested,PurposeOfLoan,CreditScore,ExistingLoansCount,LatePaymentsLastYear,LoanApproved
0,6b65a6a4-8b81-48f6-b38a-088ca65ed389,Carla Briggs,56,Male,Widowed,PhD,Employed,175074,30737,Home,557,0,6,No
1,5be6128e-18c2-4797-a142-ea7d17be3111,William Lewis,53,Male,Widowed,Other,Unemployed,100000,36560,Car,722,4,6,No
2,7412b293-4729-4739-a14f-f3d719db3ad0,Rose Jackson,19,Male,Married,Bachelor,Unemployed,220000,2264,Car,790,2,2,No


In [27]:
import numpy as np
import pandas as pd

def calculate_loan_approval(row):
    score = 0

    # Credit Score
    if row["CreditScore"] >= 750:
        score += 35
    elif row["CreditScore"] >= 700:
        score += 30
    elif row["CreditScore"] >= 650:
        score += 20
    else:
        score += 10

    # Income
    if row["AnnualIncome"] >= 100000:
        score += 25
    elif row["AnnualIncome"] >= 70000:
        score += 15
    else:
        score += 5

    # Late payments
    if row["LatePaymentsLastYear"] == 0:
        score += 20
    elif row["LatePaymentsLastYear"] <= 2:
        score += 10

    # Existing loans
    if row["ExistingLoansCount"] <= 2:
        score += 10

    # Loan burden
    ratio = row["LoanAmountRequested"] / row["AnnualIncome"]
    if ratio <= 0.3:
        score += 20
    elif ratio <= 0.5:
        score += 10

    return score


df["RiskScore"] = df.apply(calculate_loan_approval, axis=1)
df["LoanApproved"] = (df["RiskScore"] >= 60).astype(int)

In [28]:

X = df.drop(columns=["LoanApproved", "RiskScore"])
y = df["LoanApproved"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)


In [29]:

numeric_features = [
    "Age", "AnnualIncome", "LoanAmountRequested",
    "CreditScore", "ExistingLoansCount", "LatePaymentsLastYear"
]

categorical_features = [
    "Gender", "MaritalStatus", "EducationLevel",
    "EmploymentStatus", "PurposeOfLoan"
]

numeric_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="median"))
])

categorical_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("num", numeric_transformer, numeric_features),
    ("cat", categorical_transformer, categorical_features)
])

# model = RandomForestClassifier(
#     n_estimators=300,
#     max_depth=None,
#     random_state=42
# )

# model = RandomForestClassifier(
#     n_estimators=300,
#     max_depth=None,
#     min_samples_leaf=2,
#     random_state=42
# )


model = XGBClassifier(
    n_estimators=300,
    max_depth=5,
    learning_rate=0.1,
    random_state=42
)

pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", model)
])

In [30]:

pipeline.fit(X_train, y_train)


Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median'))]),
                                                  ['Age', 'AnnualIncome',
                                                   'LoanAmountRequested',
                                                   'CreditScore',
                                                   'ExistingLoansCount',
                                                   'LatePaymentsLastYear']),
                                                 ('cat',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('onehot',
                                                                   OneHotEncoder(handle_unknow...
                               feature_types=None, feature_weights=None,
                               gamma=None, grow_policy=None,
                               importance_type=None,
                               interaction_constraints=None, learning_rate=0.1,
                               max_bin=None, max_cat_threshold=None,
                               max_cat_to_onehot=None, max_delta_step=None,
                               max_depth=5, max_leaves=None,
                               min_child_weight=None, missing=nan,
                               monotone_constraints=None, multi_strategy=None,
                               n_estimators=300, n_jobs=None,
                               num_parallel_tree=None, ...))])

In [31]:

y_prob = pipeline.predict_proba(X_test)[:, 1]

y_pred = (y_prob >= 0.5).astype(int)


In [32]:
prediction = pipeline.predict(X_test)

print(type(prediction))
print(prediction[:10])

<class 'numpy.ndarray'>
[1 0 0 1 0 0 1 0 1 1]


In [122]:
# results = X_test.copy()

# results["LoanApproved_Actual"] = y_test.values
# results["Probability"] = y_prob
# results["Prediction"] = y_pred

# results["LoanApproved_Actual"] = results["LoanApproved_Actual"].map({0: "No", 1: "Yes"})
# results["Prediction"] = results["Prediction"].map({0: "No", 1: "Yes"})

In [33]:
y_prob_all = pipeline.predict_proba(X_test)

classes = pipeline.named_steps['model'].classes_
pos_index = list(classes).index(1)

y_prob = y_prob_all[:, pos_index]
y_pred = (y_prob >= 0.5).astype(int)

results = X_test.copy()
results["LoanApproved"] = y_test.values
results["Prediction"] = y_pred

results["LoanApproved"] = results["LoanApproved"].map({0: "No", 1: "Yes"})

results["Prediction"] = results["Prediction"].map({0: "No", 1: "Yes"})

results["Probability"] = y_prob


In [34]:
total_approved = (results["Prediction"] == "Yes").sum()
total_rejected = (results["Prediction"] == "No").sum()

print("Total Approved:", total_approved)
print("Total Rejected:", total_rejected)

Total Approved: 947
Total Rejected: 853


In [35]:
pd.DataFrame({
    "Actual_Loan": results["LoanApproved"],
    "Prediction": results["Prediction"]
})

,Actual_Loan,Prediction
1500,Yes,Yes
1904,No,No
3590,No,No
570,Yes,Yes
5356,No,No
...,...,...
5354,Yes,Yes
879,No,No
6963,No,No
6555,No,No


In [36]:

print(confusion_matrix(y_test, y_pred))
print(classification_report(y_test, y_pred))


[[842   7]
 [ 11 940]]
              precision    recall  f1-score   support

           0       0.99      0.99      0.99       849
           1       0.99      0.99      0.99       951

    accuracy                           0.99      1800
   macro avg       0.99      0.99      0.99      1800
weighted avg       0.99      0.99      0.99      1800



In [37]:
model = pipeline.named_steps["model"]
print(model.feature_importances_)

[0.00461087 0.21105608 0.05693292 0.18688153 0.225738   0.19141342
 0.00306567 0.         0.00334545 0.00658175 0.00605953 0.00509178
 0.00480646 0.00971272 0.00975181 0.00185465 0.00402746 0.00470559
 0.00742316 0.00887594 0.00938404 0.00281415 0.00557526 0.00953706
 0.00917419 0.00685465 0.00472586]


In [38]:
df = X_test.copy()
df["Income_to_Loan_Ratio"] = df["AnnualIncome"] / df["LoanAmountRequested"]
df["Risk_Index"] = df["CreditScore"] - (df["LatePaymentsLastYear"] * 50)

#print(df[["Income_to_Loan_Ratio", "Risk_Index"]].to_string())

In [39]:
print(df[["Income_to_Loan_Ratio", "Risk_Index"]].head())


      Income_to_Loan_Ratio  Risk_Index
1500             28.136748         566
1904             16.792833         213
3590             27.934105         337
570               5.507844         241
5356             18.853179         214


In [40]:
print(df[["Income_to_Loan_Ratio", "Risk_Index"]].describe())


       Income_to_Loan_Ratio   Risk_Index
count           1800.000000  1800.000000
mean               9.476964   347.833333
std               17.128436   212.217208
min                0.449050  -149.000000
25%                2.571145   195.000000
50%                4.389449   349.000000
75%                8.997535   503.250000
max              179.944719   837.000000


In [41]:
print("Income_to_Loan_Ratio: Minimum", df["Income_to_Loan_Ratio"].min(),
      "\nIncome_to_Loan_Ratio: Maximum", df["Income_to_Loan_Ratio"].max(),
      "\nIncome_to_Loan_Ratio: Mean",df["Income_to_Loan_Ratio"].mean())


Income_to_Loan_Ratio: Minimum 0.44905029508328803 
Income_to_Loan_Ratio: Maximum 179.9447186574531 
Income_to_Loan_Ratio: Mean 9.476964300953629


In [42]:

total_loans = len(results)
approved_loans = (results["Prediction"] == "Yes").sum()
rejected_loans = (results["Prediction"] == "No").sum()

summary = pd.DataFrame({
    "Total Loans": [total_loans],
    "Approved Loans": [approved_loans],
    "Rejected Loans": [rejected_loans]
})

summary.to_excel("Loan_Risk_Summary.xlsx", index=False)

print("Loan Risk Summary file saved successfully.")


Loan Risk Summary file saved successfully.


In [21]:
# def make_risk_statement(row):
#     dti = row["LoanAmountRequested"] / (row["AnnualIncome"] + 1)
    
#     # Hard risk signals first
#     if row["CreditScore"] < 600 or row["LatePaymentsLastYear"] > 2:
#         return "High risk due to weak credit behavior"
    
#     # Financial burden
#     elif dti > 3 or row["ExistingLoansCount"] > 3:
#         return "Moderate risk due to high debt burden"
    
#     # Soft signals (employment context)
#     elif row["EmploymentStatus"] in ["Unemployed", "Student"]:
#         return "Moderate risk due to employment status"
    
#     else:
#         return "Low risk based on current customer profile"


# def make_decision(row):
#     dti = row["LoanAmountRequested"] / (row["AnnualIncome"] + 1)

#     # Hard reject
#     if row["CreditScore"] < 600 or row["LatePaymentsLastYear"] > 2:
#         return "Reject"

#     # Impossible repayment
#     if dti > 5:
#         return "Reject"

#     # Manual review zone
#     if (dti > 3 or
#         row["ExistingLoansCount"] >= 1 or        # ← fix >= 3
#         row["CreditScore"] < 650 or              # ← borderline buffer
#         row["EmploymentStatus"] in ["Unemployed", "Student", "Retired"]):  # ← Retired added
#         return "Review manually"

#     return "Approve"


In [43]:
def make_risk_statement(row):
    """Generate business-friendly explanation for loan risk."""

    reasons = []

    # Debt-to-Income Ratio
    dti = row["LoanAmountRequested"] / (row["AnnualIncome"] + 1)

    # Credit Behaviour
    if row["CreditScore"] < 600:
        reasons.append("Low credit score")

    if row["LatePaymentsLastYear"] > 2:
        reasons.append("Frequent late payments")

    # Debt Burden
    if row["ExistingLoansCount"] >= 3:
        reasons.append("Multiple existing loans")

    if dti > 3:
        reasons.append("High debt-to-income ratio")

    # Employment
    if row["EmploymentStatus"] in ["Unemployed", "Student"]:
        reasons.append("Employment status increases repayment risk")

    # Final Statement
    if not reasons:
        return "Low risk based on current customer profile"

    return "; ".join(reasons)


def make_decision(row):
    """Business decision based on customer profile."""

    dti = row["LoanAmountRequested"] / (row["AnnualIncome"] + 1)

    # =========================
    # HARD REJECTION RULES
    # =========================
    if (
        row["CreditScore"] < 550
        or row["LatePaymentsLastYear"] >= 4
        or dti > 5
    ):
        return "Reject"

    # =========================
    # MANUAL REVIEW RULES
    # =========================
    if (
        row["CreditScore"] < 650
        or row["LatePaymentsLastYear"] >= 2
        or row["ExistingLoansCount"] >= 3
        or dti > 3
        or row["EmploymentStatus"] in ["Unemployed", "Student", "Retired"]
    ):
        return "Review Manually"

    # =========================
    # APPROVE
    # =========================
    return "Approve"

In [49]:
# df = X_test.copy()

# df["LoanApproved"] = y_test.values
# df["y_prob"] = y_prob
# df["y_pred"] = y_pred

# df["LoanApproved"] = df["LoanApproved"].replace({0: "No", 1: "Yes"})
# df["Prediction"] = df["y_pred"].replace({0: "No", 1: "Yes"})

# #df["LoanApproved"] = df["LoanApproved"].replace({0: "No", 1: "Yes"})
# #df = df.drop(columns=["LoanApprovedLabel"], errors="ignore")

# # First calculate probabilities
# df["y_prob"] = pipeline.predict_proba(X)[:, 1]

# # Then calculate predictions
# df["y_pred"] = (df["y_prob"] > 0.5).astype(int)

# df["riskstatement"] = df.apply(make_risk_statement, axis=1)
# df["decision"] = df.apply(make_decision, axis=1)

# cols = df.columns.tolist()

# # Swap y_prob and y_pred
# df["y_pred"] = df["y_pred"].replace({0: "No", 1: "Yes"})
# df["y_prob"] = pd.to_numeric(df["y_prob"], errors="coerce")
# df["y_prob"] = (df["y_prob"] * 100).round(2)

# i = cols.index("y_prob")
# j = cols.index("y_pred")
# cols[i], cols[j] = cols[j], cols[i]

# # Reorder DataFrame
# df = df[cols]

# df = df.loc[:, ~df.columns.duplicated()]

# df.rename(columns={
#     "y_pred": "Prediction",
#     "y_prob": "Probability"
# }, inplace=True)

# # Export
# df.to_excel("Loan_Risk_Assessment_AI_BusinessLogic.xlsx", index=False)

# print("\nFiles Exported Successfully")

ValueError: Length of values (9000) does not match length of index (1800)

In [22]:
# df = X_test.copy()

# # 1. Predictions from model (NEVER recalculate manually later)
# y_prob = pipeline.predict_proba(X_test)[:, 1]
# y_pred = (y_prob >= 0.5).astype(int)

# df["LoanApproved"] = y_test.values
# df["y_prob"] = y_prob
# df["y_pred"] = y_pred

# # 2. Risk logic functions (your custom logic stays here)
# df["riskstatement"] = df.apply(make_risk_statement, axis=1)
# df["decision"] = df.apply(make_decision, axis=1)

# # 3. Convert only for final display (LAST STEP)
# df["LoanApproved"] = df["LoanApproved"].replace({0: "No", 1: "Yes"})
# df["Prediction"] = df["y_pred"].replace({0: "No", 1: "Yes"})

# # 4. Format probability
# df["Probability"] = (df["y_prob"] * 100).round(2)

# # 5. Drop raw helper column if not needed
# df.drop(columns=["y_prob", "y_pred"], inplace=True)

# # 6. Export
# df.to_excel("Loan_Risk_Assessment_AI_BusinessLogic.xlsx", index=False)

# print("Files Exported Successfully")

Files Exported Successfully


In [44]:
df = X_test.copy()

# ============================================================
# 1. AI Prediction from XGBoost Model
# ============================================================
y_prob = pipeline.predict_proba(X_test)[:, 1]
y_pred = (y_prob >= 0.5).astype(int)

df["LoanApproved"] = y_test.values
df["AI Prediction"] = y_pred
df["AI Probability"] = (y_prob * 100).round(2)

# ============================================================
# 2. Risk Level based on AI Probability
# ============================================================
    
def risk_level(prob):
    if prob > 0.70:
        return "Low"
    elif prob > 0.30:
        return "Medium"
    else:
        return "High"

df["Risk Level"] = df["AI Probability"].apply(risk_level)

# ============================================================
# 3. Business Decision
# ============================================================
df["Business Decision"] = df.apply(make_decision, axis=1)

# ============================================================
# 4. Reason
# ============================================================
df["Reason"] = df.apply(make_risk_statement, axis=1)

# ============================================================
# 5. Convert numeric prediction to Yes/No
# ============================================================
df["LoanApproved"] = df["LoanApproved"].replace({
    0: "No",
    1: "Yes"
})

df["AI Prediction"] = df["AI Prediction"].replace({
    0: "No",
    1: "Yes"
})

# ============================================================
# 6. Format Probability
# ============================================================
df["AI Probability"] = df["AI Probability"].astype(str) + "%"

# ============================================================
# 7. Arrange Columns
# ============================================================
columns = [
    "CustomerID",
    "Name",
    "Age",
    "Gender",
    "MaritalStatus",
    "EducationLevel",
    "EmploymentStatus",
    "AnnualIncome",
    "LoanAmountRequested",
    "PurposeOfLoan",
    "CreditScore",
    "ExistingLoansCount",
    "LatePaymentsLastYear",
    "LoanApproved",
    "AI Prediction",
    "AI Probability",
    "Risk Level",
    "Business Decision",
    "Reason"
]

# Keep only columns that exist
columns = [c for c in columns if c in df.columns]

df = df[columns]

# ============================================================
# 8. Export
# ============================================================
df.to_excel(
    "Loan_Risk_Assessment_AI_BusinessLogic.xlsx",
    index=False
)

print("Loan Risk Assessment Report Exported Successfully")

Loan Risk Assessment Report Exported Successfully


In [24]:
import joblib

joblib.dump(pipeline, "Loan_Risk_Model.pkl")
print("Model saved successfully")



Model saved successfully


In [29]:
pip install streamlit joblib

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.
